# Imports & original datasets

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np

In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
train = pd.read_csv('UNSW15_train.csv')

In [ ]:
train.sample(7)

# Preprocessing

### Categorical features 

#### I'd argue they're these:

In [ ]:
# categorical_features = []
# for col in train.columns:
#     x = train[col].nunique()
#     if x < 40:
#         categorical_features.append(col)

#### But they could also just be these:

In [ ]:
categorical_features = ['proto', 'service', 'state', 'is_ftp_login','is_sm_ips_ports']

In [ ]:
missingvalues = train.isnull().sum().sort_values(ascending=False)
missingvalues # no missing values

### Splitting the data (by hand)

In [ ]:
def printlen(x,y):
    print("len(x) = {}, len(y) = {}".format(len(x), len(y)))

In [ ]:
X = train.drop('label', axis=1)
random_number = train['label']


In [ ]:
printlen(X, random_number)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, random_number, train_size=0.8, random_state=42)

In [ ]:
printlen(X_train, y_train)

In [ ]:
n = len(X_train)
m = len(y_train)

X_val = X_train[int(n*0.8):]
X_train = X_train[:int(n*0.8)]

y_val = y_train[int(m*0.8):]
y_train = y_train[:int(m*0.8)]


In [ ]:
len(y_train)

In [ ]:
printlen(X_train, y_train)
printlen(X_val, y_val)


In [ ]:
# scaler = StandardScaler()
# train = scaler.fit_transform(train)
# train

### Scaling the data

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


### Generate poisoned data

In [ ]:
def make_poisoned_csv(data=None, p=0.1):
    if data is None:
        return
    
    threshold_to_flip = int(p*100)
    np.random.seed(42)
    
    for x in range(len(data['label'])):
        random_number = np.random.randint(0, 100)
        value_to_flip = data.at[x, 'label']
        if random_number < threshold_to_flip:  # about probability p of a label flip
            if value_to_flip == 0:
                data.at[x, 'label'] = 1
            else: 
                data.at[x, 'label'] = 0
    
    filename = 'UNSW15_{}.csv'.format(threshold_to_flip)
    data.to_csv(filename, index=None)

In [ ]:
data = pd.read_csv('UNSW15_train.csv')

for percentage in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.5]:
    make_poisoned_csv(data, percentage)

# EDA

# Neural Network

In [ ]:
tf.random.set_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu'), 
    tf.keras.layers.Dropout(rate=0.1),
    tf.keras.layers.Dense(64, activation='relu'), 
    tf.keras.layers.Dropout(rate=0.1),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(
    loss=tf.keras.losses.binary_crossentropy,
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.F1Score(name='f1-score'),
    ]
)
history = model.fit(X_train_scaled, y_train, epochs=25, validation_data=(X_val_scaled, y_val))

In [ ]:
new_history = model.predict(X_test_scaled)

performance = model.evaluate(X_test_scaled, y_test, return_dict=True)
print(performance)


# Evaluation

In [ ]:
from matplotlib import rcParams
rcParams['figure.figsize'] = (9, 8)
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False
plt.plot(
    np.arange(1, 26), 
    history.history['loss'], label='Loss'
)
plt.plot(
    np.arange(1, 26), 
    history.history['val_loss'], label='Validation Loss'
)
# plt.plot(
#     np.arange(1, 7), 
#     history.history['precision'], label='Precision'
# )
# plt.plot(
#     np.arange(1, 7), 
#     history.history['recall'], label='Recall'
# )
plt.title('Deep Neural Network Training & Validation Loss', size=20)
plt.ylabel('Binary Cross Entropy Loss')
plt.xlabel('Epoch', size=14)

plt.legend()

In [ ]:
dt_poisoning = pd.read_csv("DT.csv")
dtb_poisoning = pd.read_csv("DTBagged.csv")
dnn_poisoning = pd.read_csv("DNN.csv")
fl_poisoning = pd.read_csv("FL.csv")

plt.plot(dt_poisoning['poisoning'], dt_poisoning['f1'], label='Decision Tree')
plt.plot(dtb_poisoning['poisoning'], dtb_poisoning['f1'], label='Decision Tree Bagged')
plt.plot(dnn_poisoning['poisoning'], dnn_poisoning['f1'], label='Deep Neural Network')
plt.plot(fl_poisoning['poisoning'], fl_poisoning['f1'], label='Federated Learning')

plt.xlabel('Amount of data poisoning (%)')
plt.ylabel('F1-Score')

plt.legend()


$H_p(q) =  -\frac{1}{N} \sum_{i=1}^N y_i\cdot\log(p(y_i)) + (1-y_i)\cdot\log(1-p(y_i)) $